[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/patterns/openai_agents_patterns.ipynb)

# Seven matched patterns with the OpenAI Agents SDK

**Task.** Answer *Which interventions reduce household food waste, and under what conditions?* while comparing seven SDK-native orchestration patterns.

The bounded catalogue contains three tutorial records: smaller plates, meal planning, and an information-only campaign. Each has a source ID, title, topics and evidence text. All framework notebooks use these records and the same versioned mock decisions, so the comparison focuses on orchestration.

**Read this notebook as:** setup → contracts/adapters → native Agent/Runner examples → matched evaluation. Runtime is under one minute on CPU; no API key, network call or model download is required.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openai-agents==0.17.8",
            "pydantic==2.12.5",
        ],
        check=True,
    )

In [ ]:
import asyncio
import json
from pathlib import Path
from typing import Any, ClassVar

from agents import Agent, Runner, function_tool, set_tracing_disabled
from agents.items import ModelResponse as SDKModelResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import (
    ResponseFunctionToolCall,
    ResponseOutputMessage,
    ResponseOutputText,
)
from pydantic import BaseModel, ConfigDict, Field

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))

from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]
TASK = "Which interventions reduce household food waste, and under what conditions?"

## What comes from where

- **OpenAI Agents SDK:** `Agent` defines instructions and tools; `Runner` owns the agent loop; `@function_tool` exposes a callable tool; handoffs transfer control to another agent; and `Agent.as_tool()` lets an orchestrator invoke specialists.
- **Pydantic:** validates every structured decision and final answer.
- **Repository:** the deterministic client, versioned response fixtures, shared message/tool contracts, and plan/critic schemas.
- **This notebook:** evidence schemas, a thin SDK response-shape adapter, bounded catalogue search, agents and evaluation checks.

The adapters translate deterministic fixture responses into SDK response items. They do not execute tools, perform handoffs, schedule parallel runs or invoke agents-as-tools; those operations remain with the SDK.

In [ ]:
# --- Notebook-defined contracts and shared deterministic fixtures ---
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2, max_length=3)
    aggregation: str = Field(pattern="^validated_union$")


class CatalogueRecord(StrictModel):
    source_id: str
    title: str
    topics: tuple[str, ...]
    evidence: str


class SearchBatch(StrictModel):
    worker: str
    records: tuple[CatalogueRecord, ...]


class WorkerAssignment(StrictModel):
    worker: str = Field(pattern="^(intervention_reviewer|planning_reviewer)$")
    query: str


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[WorkerAssignment, ...] = Field(min_length=1, max_length=2)


SearchBatch.model_rebuild(_types_namespace={"CatalogueRecord": CatalogueRecord})
OrchestrationDecision.model_rebuild(_types_namespace={"WorkerAssignment": WorkerAssignment})


def model_for(name: str) -> DeterministicMockClient:
    fixture = ScriptedScenarioFixture.model_validate(
        {
            "fixture_version": fixture_data["fixture_version"],
            "scenario": name,
            "provenance": fixture_data["provenance"],
            "responses": fixture_data["scenarios"][name],
        }
    )
    return DeterministicMockClient(fixture)


def user(text: str) -> Message:
    return Message(role=MessageRole.USER, content=text)


def search(query: str, limit: int = 3) -> list[dict]:
    terms = set(query.casefold().split())
    return [
        record for record in catalogue if terms & set(" ".join(record["topics"]).casefold().split())
    ][:limit]


tool_events: list[str] = []


@function_tool
def search_catalogue(query: str, max_results: int) -> str:
    """Search the bounded food-waste catalogue.

    Args:
        query: Search terms such as meal planning.
        max_results: Maximum number of records to return.
    """
    tool_events.append(query)
    return json.dumps(search(query, max_results))


search_contract = ToolDefinition(
    name="search_catalogue",
    description="Search the bounded food-waste catalogue.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)

In [ ]:
# --- Shape adapters: shared mock response -> SDK response item ---
def output_message(response_id: str, value: Any) -> ResponseOutputMessage:
    return ResponseOutputMessage(
        id=response_id,
        content=[
            ResponseOutputText(
                annotations=[],
                text=json.dumps(value, sort_keys=True),
                type="output_text",
                logprobs=[],
            )
        ],
        role="assistant",
        status="completed",
        type="message",
    )


def function_call(call_id: str, name: str, arguments: dict) -> ResponseFunctionToolCall:
    return ResponseFunctionToolCall(
        arguments=json.dumps(arguments),
        call_id=call_id,
        name=name,
        type="function_call",
        id=call_id,
        status="completed",
    )


class FixtureSDKModel(SDKModel):
    """Expose one versioned repository scenario through the SDK Model interface."""

    def __init__(self, scenario: str):
        self.scenario = scenario
        self.core = model_for(scenario)
        self.calls = 0

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        use_search_tool = self.scenario == "react" and bool(tools)
        response = await self.core.generate(
            [user(str(input))],
            tools=[search_contract] if use_search_tool else (),
        )
        self.calls += 1

        if response.tool_calls:
            items = [
                function_call(call.call_id, call.name, call.arguments)
                for call in response.tool_calls
            ]
        elif self.scenario == "routing" and handoffs:
            route = response.structured_output["route"]
            target = next(handoff for handoff in handoffs if route in handoff.tool_name)
            items = [function_call(response.response_id, target.tool_name, {})]
        elif self.scenario == "orchestrator_worker" and self.calls == 1 and tools:
            tool_names = {tool.name for tool in tools}
            items = [
                function_call(
                    f"{response.response_id}-{index}",
                    assignment["worker"],
                    {"input": assignment["query"]},
                )
                for index, assignment in enumerate(
                    response.structured_output["assignments"], start=1
                )
                if assignment["worker"] in tool_names
            ]
        else:
            items = [output_message(response.response_id, response.structured_output)]

        return SDKModelResponse(
            output=items,
            usage=Usage(),
            response_id=response.response_id,
        )

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


class CatalogueWorkerSDKModel(SDKModel):
    """Return typed evidence for an SDK worker or handoff target."""

    def __init__(self, worker: str, query: str):
        self.worker = worker
        self.query = query
        self.calls = 0

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        self.calls += 1
        batch = SearchBatch(worker=self.worker, records=tuple(search(self.query, 2)))
        return SDKModelResponse(
            output=[output_message(f"{self.worker}-{self.calls}", batch.model_dump())],
            usage=Usage(),
            response_id=f"{self.worker}-{self.calls}",
        )

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None

## 1–2. Prompt chaining and routing

Prompt chaining runs two typed agents in sequence, with the first result becoming the second input. Routing uses an SDK handoff: the triage model emits the selected handoff tool call, and `Runner` transfers execution to the specialist.

In [ ]:
# --- 1. Prompt chaining: typed Agent stages run by Runner ---
chain_model = FixtureSDKModel("prompt_chaining")
extractor = Agent(
    name="Extractor",
    instructions="Extract one supported claim and its source ID.",
    model=chain_model,
    output_type=Claim,
)
claim_result = await Runner.run(extractor, TASK, max_turns=2)
claim = claim_result.final_output

synthesiser = Agent(
    name="Synthesiser",
    instructions="Use only the supplied validated claim.",
    model=chain_model,
    output_type=Answer,
)
chain_result = await Runner.run(
    synthesiser,
    json.dumps(claim.model_dump()),
    max_turns=2,
)
chain_state = {
    "answer": chain_result.final_output.model_dump(),
    "trace": ["Runner:Extractor", "typed input", "Runner:Synthesiser"],
    "stop": "criteria_met",
}


# --- 2. Routing: Runner executes a native handoff ---
research_model = CatalogueWorkerSDKModel(
    "research_specialist",
    "meal planning portion",
)
research_agent = Agent(
    name="Research specialist",
    instructions="Return bounded catalogue evidence.",
    model=research_model,
    output_type=SearchBatch,
)
clarify_agent = Agent(
    name="Clarifier",
    instructions="Explain which scope is missing.",
    model=CatalogueWorkerSDKModel("clarifier", "unmatched"),
    output_type=SearchBatch,
)
triage_agent = Agent(
    name="Triage",
    instructions="Transfer the request to the appropriate specialist.",
    model=FixtureSDKModel("routing"),
    handoffs=[research_agent, clarify_agent],
)
route_result = await Runner.run(triage_agent, TASK, max_turns=3)
route_batch = route_result.final_output
route_state = {
    "results": list(route_batch.records),
    "trace": ["Runner:Triage", "handoff", f"Agent:{route_result.last_agent.name}"],
    "stop": (
        "criteria_met" if route_result.last_agent.name == "Research specialist" else "safe_fallback"
    ),
}

## 3–5. Parallelisation, ReAct and planner–executor

Parallelisation launches independent `Runner.run()` calls concurrently. ReAct gives an agent a real function tool; the SDK loop executes it and returns the observation before the final answer. Planning uses a typed planner agent followed by an explicit dependency gate because the SDK has no graph scheduler.

In [ ]:
# --- 3. Parallelisation: concurrent native Runner calls ---
parallel_planner = Agent(
    name="Parallel planner",
    instructions="Propose two independent bounded searches.",
    model=FixtureSDKModel("parallelisation"),
    output_type=ParallelDecision,
)
parallel_plan = (await Runner.run(parallel_planner, TASK, max_turns=2)).final_output

parallel_workers = [
    Agent(
        name=f"Parallel worker {index}",
        instructions="Return evidence for exactly one assigned query.",
        model=CatalogueWorkerSDKModel(f"parallel_worker_{index}", query),
        output_type=SearchBatch,
    )
    for index, query in enumerate(parallel_plan.queries, start=1)
]
parallel_results = await asyncio.gather(
    *(
        Runner.run(worker, query, max_turns=2)
        for worker, query in zip(parallel_workers, parallel_plan.queries, strict=True)
    )
)
parallel_batches = [result.final_output for result in parallel_results]
parallel_source_ids = sorted(
    {record.source_id for batch in parallel_batches for record in batch.records}
)
parallel_state = {
    "source_ids": parallel_source_ids,
    "trace": ["Runner fan-out", "asyncio.gather", "validated_union"],
    "stop": "criteria_met",
}


# --- 4. ReAct: Agent + function_tool + Runner tool loop ---
react_agent = Agent(
    name="ReAct researcher",
    instructions="Use search_catalogue, then answer with cited evidence.",
    model=FixtureSDKModel("react"),
    tools=[search_catalogue],
    output_type=Answer,
)
react_result = await Runner.run(react_agent, TASK, max_turns=3)
react_state = {
    "answer": react_result.final_output.model_dump(),
    "trace": ["Agent action", "@function_tool execution", "Agent finish"],
    "steps": react_agent.model.calls,
    "stop": "criteria_met" if tool_events else "tool_not_called",
}


# --- 5. Planner-executor: typed Agent plan + dependency gate ---
planner_agent = Agent(
    name="Planner",
    instructions="Create a three-step dependency-aware research plan.",
    model=FixtureSDKModel("planner_executor"),
    output_type=PlanDecision,
)
plan = (await Runner.run(planner_agent, TASK, max_turns=2)).final_output
plan_valid = len(plan.steps) == 3 and all(
    plan.steps[index].depends_on == (plan.steps[index - 1].step_id,) for index in (1, 2)
)
plan_state = {
    "plan": plan.model_dump(),
    "trace": ["Runner:Planner", "dependency_executor" if plan_valid else "dependency_stop"],
    "stop": "criteria_met" if plan_valid else "invalid_plan",
}

## 6–7. Critic–reviser and orchestrator–worker

Writer, critic and reviser are separate typed agents, with one explicit revision budget. The orchestrator exposes two specialist agents through `Agent.as_tool()`; `Runner` executes both delegated calls and feeds their outputs back for synthesis.

In [ ]:
# --- 6. Critic-reviser: distinct typed Agents and one revision gate ---
critic_model = FixtureSDKModel("critic_reviser")
writer_agent = Agent(
    name="Writer",
    instructions="Draft an evidence answer.",
    model=critic_model,
    output_type=Answer,
)
critic_agent = Agent(
    name="Critic",
    instructions="Check support and stated conditions.",
    model=critic_model,
    output_type=CriticDecision,
)
reviser_agent = Agent(
    name="Reviser",
    instructions="Revise once using the critic issues.",
    model=critic_model,
    output_type=Answer,
)
draft = (await Runner.run(writer_agent, TASK, max_turns=2)).final_output
critique = (
    await Runner.run(
        critic_agent,
        json.dumps(draft.model_dump()),
        max_turns=2,
    )
).final_output
revisions = 0
if not critique.accepted:
    draft = (
        await Runner.run(
            reviser_agent,
            json.dumps({"draft": draft.model_dump(), "issues": critique.issues}),
            max_turns=2,
        )
    ).final_output
    revisions = 1
critic_state = {
    "answer": draft.model_dump(),
    "revisions": revisions,
    "trace": ["Runner:Writer", "Runner:Critic", "Runner:Reviser"],
    "stop": "criteria_met" if draft.source_ids else "revision_budget_exhausted",
}


# --- 7. Orchestrator-worker: specialist Agents exposed as tools ---
intervention_model = CatalogueWorkerSDKModel(
    "intervention_reviewer",
    "smaller plates",
)
planning_model = CatalogueWorkerSDKModel(
    "planning_reviewer",
    "meal planning",
)
intervention_agent = Agent(
    name="Intervention reviewer",
    instructions="Review intervention evidence only.",
    model=intervention_model,
    output_type=SearchBatch,
)
planning_agent = Agent(
    name="Planning reviewer",
    instructions="Review planning evidence only.",
    model=planning_model,
    output_type=SearchBatch,
)
orchestrator_agent = Agent(
    name="Orchestrator",
    instructions="Delegate to both specialists, then synthesise their evidence.",
    model=FixtureSDKModel("orchestrator_worker"),
    tools=[
        intervention_agent.as_tool(
            tool_name="intervention_reviewer",
            tool_description="Review smaller-plate evidence.",
        ),
        planning_agent.as_tool(
            tool_name="planning_reviewer",
            tool_description="Review meal-planning evidence.",
        ),
    ],
    output_type=Answer,
)
orchestrator_result = await Runner.run(
    orchestrator_agent,
    TASK,
    max_turns=4,
)
worker_answer = orchestrator_result.final_output
orchestrator_state = {
    "answer": worker_answer.model_dump(),
    "worker_count": intervention_model.calls + planning_model.calls,
    "trace": ["Runner:Orchestrator", "Agent.as_tool workers", "synthesis"],
    "stop": "criteria_met",
}

## Evaluation summary

Each pattern must terminate and meet the same observable check used by the other framework notebooks.

Limitations: deterministic adapters exercise SDK orchestration rather than hosted-model judgment; handoffs and tool loops still need production budgets; and the pinned SDK/OpenAI package combination is version-sensitive.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["stop"] == "criteria_met",
    "routing": route_state["stop"] == "criteria_met" and len(route_state["results"]) == 2,
    "parallelisation": parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["stop"] == "criteria_met" and react_state["steps"] <= 2,
    "planner_executor": plan_state["stop"] == "criteria_met",
    "critic_reviser": critic_state["revisions"] == 1,
    "orchestrator_worker": orchestrator_state["worker_count"] == 2,
}
pattern_limitations = {
    "prompt_chaining": "early errors propagate",
    "routing": "valid handoffs can still be semantically wrong",
    "parallelisation": "branches can duplicate work",
    "react": "tool loops require strict budgets",
    "planner_executor": "invalid dependencies stop execution",
    "critic_reviser": "one revision may be insufficient",
    "orchestrator_worker": "worker findings can conflict",
}
assert set(pattern_limitations) == set(pattern_evaluations)
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "OpenAI Agents SDK",
    "evaluation": pattern_evaluations,
    "native_abstractions": [
        "Agent",
        "Runner",
        "function_tool",
        "handoffs",
        "Agent.as_tool",
    ],
    "limitations": pattern_limitations,
}